# Financial Analysis Demo with Cash

This notebook demonstrates the capabilities of the `cash` library for caching expensive data science operations. We will work with a large, deterministically generated financial dataset.

In [1]:
import cash
import pandas as pd
import numpy as np
import time
import os
print(os.getcwd())
#os.chdir(r'C:\Users\Philipp\My Drive\Cloud\PycharmProjects\cash\examples')

c:\Users\Philipp\Downloads\oldpc\nvme\Users\Philipp\My Drive\Cloud\PycharmProjects\cash\examples


In [2]:
%cash_on
%cash_debug on

✅ Cash enabled. Your computations will be cached automatically.
   Run %cash_help for available commands.
   Found existing cache with 45 entries.
[Tip] Cash reads upstream cells from the saved notebook file.
   Save (Ctrl+S) after editing upstream cells, or enable auto-save:
   Settings -> "files.autoSave": "afterDelay"
Cache debug output enabled.


## 1. Data Loading
Loading a large CSV file can be slow. With `cash`, this operation is cached after the first run.

In [6]:
print(os.getcwd())
# Ensure the data exists (it should have been generated by generate_financial_data.py)
data_path = 'large_financial_data.csv'
#if not os.path.exists(data_path):
    #print("Data file not found! Please run generate_financial_data.py first.")
#else:
print("Loading data...")
# This read_csv call will be cached
df = pd.read_csv(data_path)
df['Date'] = pd.to_datetime(df['Date'])
print(df.head())

[PROXY_CELL_ID] Captured cell_id early: vscode-notebook-cell:/c%3A/Users/Philipp/Downloads/oldpc/nvme/Users/Philipp/My%20Drive/Cloud/PycharmProjects/cash/examples/financial_analysis_demo.ipynb#W4sZmlsZQ%3D%3D


[TIMING_PROXY] Start cached_run_cell: 17:20:14.679063
[ENSURE_STATE_DEBUG] Cell code: print(os.getcwd())
# Ensure the data exists (it sh...
[ENSURE_STATE_DEBUG] Analyzed inputs: {'print', 'os', 'pd'}
[ENSURE_STATE_DEBUG] Analyzed outputs: {'df', 'data_path'}
[ENSURE_STATE_DEBUG] Current user_ns keys (first 10): ['__name__', '__doc__', '__package__', '__loader__', '__spec__', '__builtin__', '__builtins__', '_ih', '_oh', '_dh']
[STATE] 'print' not in cache, but found in built-ins. Using built-in.
[TIMING_PROXY] Ensure state: 3.61ms
[TIMING_PROXY] Total restore time: 0.01ms
[TIMING_PROXY] Total execution time: 0.00ms
[TIMING_PROXY] Pure overhead (excl. restore+exec): 3.63ms
[TIMING_PROXY] Start executing statements...
c:\Users\Philipp\Downloads\oldpc\nvme\Users\Philipp\My Drive\Cloud\PycharmProjects\cash\examples
Loading data...


ModuleNotFoundError: No module named 'pyarrow'

[CELL_ID] Captured cell_id: vscode-notebook-cell:/c%3A/Users/Philipp/Downloads/oldpc/nvme/Users/Philipp/My%20Drive/Cloud/PycharmProjects/cash/examples/financial_analysis_demo.ipynb#W4sZmlsZQ%3D%3D


## 2. Preprocessing
Basic sorting and cleaning.

In [ ]:
print("Sorting data...")
t0 = time.time()
df = df.sort_values(by=['Ticker', 'Date'])
print(f"Sorted in {time.time() - t0:.2f}s")

In [3]:
df #print

[PROXY_CELL_ID] Captured cell_id early: vscode-notebook-cell:/c%3A/Users/Philipp/My%20Drive/Cloud/PycharmProjects/cash/examples/financial_analysis_demo.ipynb#X10sZmlsZQ%3D%3D


[TIMING_PROXY] Start cached_run_cell: 17:35:42.243486
[ENSURE_STATE_DEBUG] Cell code: df #print...
[ENSURE_STATE_DEBUG] Analyzed inputs: {'df'}
[ENSURE_STATE_DEBUG] Analyzed outputs: set()
[ENSURE_STATE_DEBUG] Current user_ns keys (first 10): ['__name__', '__doc__', '__package__', '__loader__', '__spec__', '__builtin__', '__builtins__', '_ih', '_oh', '_dh']
[STATE] Cannot restore 'df': no cached source found
[STATE] Could not restore 'df' from cache. Hoping for upstream re-execution.
[CACHE_KEY] Code: import cash... | source_hash: 336a7997fb44... | input_hashes: [] | func: (none) | module: (none) | cache_key: stmt:016d2a26573ceacb5a1d73f17...
[CACHE_KEY] Code: import pandas as pd... | source_hash: 8c0859a9b118... | input_hashes: [] | func: (none) | module: (none) | cache_key: stmt:d757c3e4af62cef4a7dcfe2a0...
[CACHE_KEY] Code: import numpy as np... | source_hash: 6f11581db2b4... | input_hashes: [] | func: (none) | module: (none) | cache_key: stmt:d09267300d1a162f3c43474d5...
[CACHE_KEY

,Date,Ticker,Open,High,Low,Close,Volume
0,2020-01-01 00:00:00,AAPL,100.496814,100.835158,98.367369,99.389654,611180
5,2020-01-01 00:05:00,AAPL,102.061478,104.873044,98.079078,101.554832,935078
10,2020-01-01 00:10:00,AAPL,104.018293,106.607716,103.477609,103.337525,644047
15,2020-01-01 00:15:00,AAPL,99.594540,100.426509,99.107421,98.537201,704594
20,2020-01-01 00:20:00,AAPL,98.041778,98.792337,97.172835,98.220357,558122
...,...,...,...,...,...,...,...
1000014,2021-11-25 10:39:00,TSLA,1.000000,4.563497,0.929661,0.026647,778156
1000015,2021-11-25 10:39:00,TSLA,1.000000,4.563497,0.929661,0.026647,778156
1000016,2021-11-25 10:39:00,TSLA,1.000000,4.563497,0.929661,0.026647,778156
1000017,2021-11-25 10:39:00,TSLA,1.000000,4.563497,0.929661,0.026647,778156


[CELL_ID] Captured cell_id: vscode-notebook-cell:/c%3A/Users/Philipp/My%20Drive/Cloud/PycharmProjects/cash/examples/financial_analysis_demo.ipynb#X10sZmlsZQ%3D%3D


## 3. Heavy Computation (Statement-wise Caching)
Here we perform multiple heavy calculations using custom rolling window functions. These are significantly slower than vectorized pandas operations, making them perfect candidates for caching. `cash` caches these statement-wise.

**Try this:** Run the cell once. Then change the window size in the first statement (SMA) and run it again. You'll see the second statement (Voladj) loads instantly from cache!

In [3]:
print("Calculating Volatility Adjusted Mean (Statement 1)....")
t0 = time.time()
# Heavy operation 1: Another slow rolling application
df['VolAdj_20'] = df.groupby('Ticker')['Close'].transform(lambda x: x.rolling(window=20).apply(lambda y: np.mean(y) / (np.std(y) + 1e-6), raw=True))
print(f"VolAdj calculated in {time.time() - t0:.2f}s")

print("Calculating Weighted SMA (Statement 2)...")
t0 = time.time()
# Heavy operation 2: Custom weighted mean using apply() (slow)
def custom_weighted_mean(x):
    weights = np.arange(1, len(x) + 1)
    return np.sum(x * weights) / np.sum(weights)

df['SMA_71'] = df.groupby('Ticker')['Close'].transform(lambda x: x.rolling(window=50).apply(custom_weighted_mean, raw=True))
print(f"SMA calculated  in {time.time() - t0:.2f}s")


df

[PROXY_CELL_ID] Captured cell_id early: vscode-notebook-cell:/c%3A/Users/Philipp/Downloads/oldpc/nvme/Users/Philipp/My%20Drive/Cloud/PycharmProjects/cash/examples/financial_analysis_demo.ipynb#X12sZmlsZQ%3D%3D


[TIMING_PROXY] Start cached_run_cell: 17:26:16.381395
[ENSURE_STATE_DEBUG] Cell code: print("Calculating Volatility Adjusted Mean (State...
[ENSURE_STATE_DEBUG] Analyzed inputs: {'np', 'print', 'len', 'df', 'time'}
[ENSURE_STATE_DEBUG] Analyzed outputs: {'custom_weighted_mean', 'df', 't0'}
[ENSURE_STATE_DEBUG] Current user_ns keys (first 10): ['__name__', '__doc__', '__package__', '__loader__', '__spec__', '__builtin__', '__builtins__', '_ih', '_oh', '_dh']
[STATE] 'print' not in cache, but found in built-ins. Using built-in.
[STATE] 'len' not in cache, but found in built-ins. Using built-in.
[STATE] Cannot restore 'df': no cached source found
[STATE] Could not restore 'df' from cache. Hoping for upstream re-execution.
[CACHE_KEY] Code: import cash... | source_hash: 336a7997fb44... | input_hashes: [] | func: (none) | module: (none) | cache_key: stmt:016d2a26573ceacb5a1d73f17...
[CACHE_KEY] Code: import pandas as pd... | source_hash: 8c0859a9b118... | input_hashes: [] | func: (none) | m

,Date,Ticker,Open,High,Low,Close,Volume,VolAdj_20,SMA_71
0,2020-01-01 00:00:00,AAPL,100.496814,100.835158,98.367369,99.389654,611180,NaN,NaN
5,2020-01-01 00:05:00,AAPL,102.061478,104.873044,98.079078,101.554832,935078,NaN,NaN
10,2020-01-01 00:10:00,AAPL,104.018293,106.607716,103.477609,103.337525,644047,NaN,NaN
15,2020-01-01 00:15:00,AAPL,99.594540,100.426509,99.107421,98.537201,704594,NaN,NaN
20,2020-01-01 00:20:00,AAPL,98.041778,98.792337,97.172835,98.220357,558122,NaN,NaN
...,...,...,...,...,...,...,...,...,...
1000014,2021-11-25 10:39:00,TSLA,1.000000,4.563497,0.929661,0.026647,778156,0.430843,0.511836
1000015,2021-11-25 10:39:00,TSLA,1.000000,4.563497,0.929661,0.026647,778156,0.351808,0.484707
1000016,2021-11-25 10:39:00,TSLA,1.000000,4.563497,0.929661,0.026647,778156,0.276829,0.459274
1000017,2021-11-25 10:39:00,TSLA,1.000000,4.563497,0.929661,0.026647,778156,0.276968,0.435355


[CELL_ID] Captured cell_id: vscode-notebook-cell:/c%3A/Users/Philipp/Downloads/oldpc/nvme/Users/Philipp/My%20Drive/Cloud/PycharmProjects/cash/examples/financial_analysis_demo.ipynb#X12sZmlsZQ%3D%3D


In [ ]:
#print(x)
df

## 4. More Metrics
Adding RSI calculation in a separate cell.

In [ ]:
def calculate_rsi(data, window=14):
    delta = data.diff()
    gain = (delta.where(delta > 0, 0)).rolling(window=window).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(window=window).mean()
    rs = gain / loss
    return 100 - (100 / (1 + rs))

print("Calculating RSI...")
t0 = time.time()
df['RSI'] = df.groupby('Ticker')['Close'].transform(calculate_rsi)
print(f"RSI calculated in {time.time() - t0:.2f}s")
df

## 5. Aggregation & Analysis

In [ ]:
summary = df.groupby('Ticker').agg({
    'Close': ['mean', 'std'],
    'Volume': 'sum',
    'RSI': 'mean',
    'SMA_50': 'last'
})
print(summary)

In [ ]:
print("hi")
print(df)

## 6. Loop Caching Demo

Cash can now cache **individual iterations** of loops! Each iteration is cached separately based on:
- The loop variable value
- Dependencies accessed in that iteration

This means if you change earlier iterations, later iterations that are independent can still be restored from cache.

In [13]:
# Processing each ticker separately in a loop
# Each iteration is cached independently!
ticker_stats = {}
print(df)
a = []

for ticker in ["TSLA", "GOOGL", "AAPL", "AMZN", "MSFT", "NFLX", "xx"]:
    ticker_data = df[df["Ticker"] == ticker]
    stats = {
        "mean_close": [ticker_data["Close"].mean() for i in range(10000)],
        "std_close": ticker_data["Close"].std(),
        "min_volume": ticker_data["Volume"].min(),
        "max_volume": ticker_data["Volume"].max()
    }
    ticker_stats[ticker] = stats
    print(f"{ticker}: mean={stats['mean_close'][0]:.2f}, std={stats['std_close']:.2f}")
    a.append(ticker)

print(ticker_stats.keys())
print("\nDone processing all tickers!")


[PROXY_CELL_ID] Captured cell_id early: vscode-notebook-cell:/c%3A/Users/Philipp/My%20Drive/Cloud/PycharmProjects/cash/examples/financial_analysis_demo.ipynb#X23sZmlsZQ%3D%3D


TYPE,CONTENT,STORAGE,TIME
⚡ Restored,ticker_statsticker_stats = {},← RAM,Saved 0.00s
⚡ Restored,No outputsprint(df),← RAM,Saved 0.01s
⚡ Restored,aa = [],← RAM,Saved 0.00s
⚙️ Loop,"▶ 🔁 ticker ∈ [TSLA, GOOGL, AAPL, …, xx] (All 7 computed, 5 stmts)",→ RAM,1.19s (Saved 24.13s)
⚡,▸ ticker_data = df[df['Ticker'] == ticker] (all cached),← RAM,0.03s (↑0.2s)
⚡,cached — ticker=TSLA,← RAM,0.006s (↑0.0s)
⚡,cached — ticker=GOOGL,← RAM,0.005s (↑0.0s)
⚡,cached — ticker=AAPL,← RAM,0.006s (↑0.0s)
⚡,cached — ticker=AMZN,← RAM,0.005s (↑0.0s)
⚡,cached — ticker=MSFT,← RAM,0.006s (↑0.0s)


[TIMING_PROXY] Start cached_run_cell: 17:09:43.426763
[ENSURE_STATE_DEBUG] Cell code: # Processing each ticker separately in a loop
# Ea...
[ENSURE_STATE_DEBUG] Analyzed inputs: {'range', 'print', 'df'}
[ENSURE_STATE_DEBUG] Analyzed outputs: {'ticker', 'stats', 'ticker_stats', 'ticker_data', 'a'}
[ENSURE_STATE_DEBUG] Current user_ns keys (first 10): ['__name__', '__doc__', '__package__', '__loader__', '__spec__', '__builtin__', '__builtins__', '_ih', '_oh', '_dh']
[STATE] 'range' not in cache, but found in built-ins. Using built-in.
[STATE] 'print' not in cache, but found in built-ins. Using built-in.
[TIMING_PROXY] Ensure state: 311.86ms
[TIMING_PROXY] Total restore time: 0.02ms
[TIMING_PROXY] Total execution time: 0.00ms
[TIMING_PROXY] Pure overhead (excl. restore+exec): 312.28ms
[TIMING_PROXY] Start executing statements...
                       Date Ticker        Open        High         Low  \
0       2020-01-01 00:00:00   AAPL  100.496814  100.835158   98.367369   
5       2020-0

In [6]:
ticker_data

[PROXY_CELL_ID] Captured cell_id early: vscode-notebook-cell:/c%3A/Users/Philipp/My%20Drive/Cloud/PycharmProjects/cash/examples/financial_analysis_demo.ipynb#X24sZmlsZQ%3D%3D


[TIMING_PROXY] Start cached_run_cell: 17:09:09.290558
[ENSURE_STATE_DEBUG] Cell code: ticker_data...
[ENSURE_STATE_DEBUG] Analyzed inputs: {'ticker_data'}
[ENSURE_STATE_DEBUG] Analyzed outputs: set()
[ENSURE_STATE_DEBUG] Current user_ns keys (first 10): ['__name__', '__doc__', '__package__', '__loader__', '__spec__', '__builtin__', '__builtins__', '_ih', '_oh', '_dh']
[CACHE_KEY] Input 'a' using virtual_lineage: 836c3cb002ab8b7e...
[CACHE_KEY] Input 'df' using virtual_lineage: ee776483bb391b52...
[CACHE_KEY] Input 'ticker_stats' using virtual_lineage: 93ee8ec19d051cc9...
[CACHE_KEY] Code: for ticker in ['TSLA', 'GOOGL', 'AAPL', 'AMZN', 'MSFT', 'NFLX', 'xx']:
    ticke... | source_hash: c6c16f72a599... | input_hashes: ['836c3cb002ab...', 'ee776483bb39...', '93ee8ec19d05...'] | func: (none) | module: (none) | cache_key: stmt:54c5396bdee8034a7a54215b3...
TSLA: mean=78.26, std=120.39
GOOGL: mean=78.27, std=120.39
AAPL: mean=78.26, std=120.40
AMZN: mean=78.27, std=120.40
MSFT: mean=78.27, s

,Date,Ticker,Open,High,Low,Close,Volume,VolAdj_20,SMA_70,RSI


[CELL_ID] Captured cell_id: vscode-notebook-cell:/c%3A/Users/Philipp/My%20Drive/Cloud/PycharmProjects/cash/examples/financial_analysis_demo.ipynb#X24sZmlsZQ%3D%3D


In [5]:
{
        "mean_close": [ticker_data["Close"].mean() for i in range(10000)],
        "std_close": ticker_data["Close"].std(),
        "min_volume": ticker_data["Volume"].min(),
        "max_volume": ticker_data["Volume"].max()
}

[PROXY_CELL_ID] Captured cell_id early: vscode-notebook-cell:/c%3A/Users/Philipp/My%20Drive/Cloud/PycharmProjects/cash/examples/financial_analysis_demo.ipynb#Y114sZmlsZQ%3D%3D


[TIMING_PROXY] Start cached_run_cell: 14:15:37.209718
[ENSURE_STATE_DEBUG] Cell code: {
        "mean_close": [ticker_data["Close"].mean...
[ENSURE_STATE_DEBUG] Analyzed inputs: {'ticker_data', 'range'}
[ENSURE_STATE_DEBUG] Analyzed outputs: set()
[ENSURE_STATE_DEBUG] Current user_ns keys (first 10): ['__name__', '__doc__', '__package__', '__loader__', '__spec__', '__builtin__', '__builtins__', '_ih', '_oh', '_dh']
[STATE] 'range' not in cache, but found in built-ins. Using built-in.
[TIMING_PROXY] Ensure state: 9.49ms
[TIMING_PROXY] Total restore time: 0.01ms
[TIMING_PROXY] Total execution time: 0.00ms
[TIMING_PROXY] Pure overhead (excl. restore+exec): 9.53ms
[TIMING_PROXY] Start executing statements...
[TIMING_PROXY] PROXY TOTAL: 424.3ms
[TIMING_PROXY] Badge init: 6.4ms
[TIMING_PROXY] Upstream check: 9.5ms
[TIMING_PROXY] Badge progress renders: 10.3ms


{'mean_close': [nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,
  nan,

[CELL_ID] Captured cell_id: vscode-notebook-cell:/c%3A/Users/Philipp/My%20Drive/Cloud/PycharmProjects/cash/examples/financial_analysis_demo.ipynb#Y114sZmlsZQ%3D%3D


In [ ]:
ticker_stats.keys()
a

### Conditional Caching

Cash also caches **branches of if statements** - only the executed branch is stored, making cache keys more precise.

In [ ]:
# Conditional processing based on data size
row_count = len(df)

if row_count > 500000:
    # Heavy processing for large datasets
    sample = df.sample(n=10000, random_state=42)
    report_type = "sampled"
    print(f"Large dataset ({row_count:,} rows) - using sampled analysis")
else:
    # Full processing for smaller datasets
    sample = df.copy()
    report_type = "full"
    print(f"Small dataset ({row_count:,} rows) - using full analysis")

print(f"Report type: {report_type}, sample size: {len(sample):,}")

### Nested Loops Example

Even nested loops work - each combination of outer/inner loop variables gets its own cache entry.

In [ ]:
# Compute metrics for multiple tickers across different time windows
windows = [10, 20, 50]
tickers = ["AAPL", "GOOGL"]

results = {}
for ticker in tickers:
    results[ticker] = {}
    ticker_data = df[df["Ticker"] == ticker]["Close"]
    for window in windows:
        sma = ticker_data.rolling(window=window).mean().iloc[-1]
        results[ticker][f"SMA_{window}"] = sma
        print(f"{ticker} SMA-{window}: {sma:.2f}")

print("\nAll window calculations complete!")

In [ ]:
for a, b in zip([1, 2, 3, 4, 5, 6, 7], ['x', 'y', 'z', 'u', 'v', 'w', 't']):
    print(f"{a} - {b}")

In [ ]:
try:
    print("hi")
    asdf = 235
    raise ValueError("This is a test error")
    x = 123
except ValueError as e:
    print(f"Value error occurred: {e}")
    time.sleep(1)  # Simulate some cleanup time

In [ ]:
c = 122

In [ ]:

def f(a):
    return c + a

x = {'b': 123}
x['a'] = f(0)
print(x)

In [ ]:
x

In [ ]:
import sys
sys.path.append("examples")
import metrics
print("Testing metrics module...")
print(metrics.increment(5))

In [ ]:
class Test:
    def __init__(self):
        self.x = 1
        self.y = 2

a = Test()
print(a.x)

a.x = 126

print(a.x)

In [ ]:
a.x = 1290

In [ ]:
print(a.x)
a.x

In [ ]:
print("hi")
raise ValueError("This is a test error to demonstrate error handling in the notebook.")
print("ho")

In [7]:
@cash.cache
def dep(a):
    import time
    time.sleep(1)
    return a + 1

[PROXY_CELL_ID] Captured cell_id early: vscode-notebook-cell:/c%3A/Users/Philipp/My%20Drive/Cloud/PycharmProjects/cash/examples/financial_analysis_demo.ipynb#Y162sZmlsZQ%3D%3D


[TIMING_PROXY] Start cached_run_cell: 17:37:09.114049
[ENSURE_STATE_DEBUG] Cell code: @cash.cache
def dep(a):
    import time
    time.s...
[ENSURE_STATE_DEBUG] Analyzed inputs: {'cash'}
[ENSURE_STATE_DEBUG] Analyzed outputs: {'dep'}
[ENSURE_STATE_DEBUG] Current user_ns keys (first 10): ['__name__', '__doc__', '__package__', '__loader__', '__spec__', '__builtin__', '__builtins__', '_ih', '_oh', '_dh']
[CACHE_KEY] Output module lineage for 'cash': 69ef73c71132...
[CACHE_KEY] Code: import cash... | source_hash: 336a7997fb44... | input_hashes: [] | func: (none) | module: :out:cash:69ef73c71132298b82a5 | cache_key: stmt:69e9540237adffbf72f60b88b...
[CACHE_KEY] Output module lineage for 'pd': d757c3e4af62...
[CACHE_KEY] Code: import pandas as pd... | source_hash: 8c0859a9b118... | input_hashes: [] | func: (none) | module: :out:pd:d757c3e4af62cef4a7dcfe | cache_key: stmt:cd8c8894d97cd656145e7cbce...
[CACHE_KEY] Output module lineage for 'np': d09267300d1a...
[CACHE_KEY] Code: import numpy as

In [ ]:
@cash.cache
def fun(a, b):
    import time
    time.sleep(1)
    return a + b + dep(a)

x = 6
[fun(x, i % 3) for i in range(51)]

In [ ]:
import metrics

print(metrics.super_fun(10))
sum([metrics.fun(x, i % 3) for i in range(51)])

In [ ]:
@cash.cache
def super_fun(df):
    import time
    time.sleep(1)  # Simulate a heavy operation
    return df.iloc[0]

In [ ]:
df = pd.DataFrame({
    'A': [1, 2, 3],
    'B': [4, 5, 6]
})
print(super_fun(df), "", "")